<div align="center">

# Encode a file and pick up where you stopped

Turn a text file into vectors and continue after an interruption.

</div>

## When do you need this?

You have an export of reviews, articles, or support messages.
The file may be too large for one in-memory matrix, or the job may stop halfway through.
**Save completed work** so a restart can continue from the first missing row.

## The story in pictures

![A text file feeds small batches into fse and a local vector store. A restart skips rows already saved.](media/03-files-with-resume.svg)

## 1. Prepare a real text file

We use 1,000 articles from the [AG News test split](https://huggingface.co/datasets/SetFit/ag_news) and POTION-base-8M.
**One line owns one row**, including an empty line.
The [shared setup](README.md#run-a-recipe) installs the pinned packages and explains the temporary workspace.

In [1]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

Recipe run: 2026-11-01. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


Load POTION-base-8M and read a few real articles.
Print an article and the vector size to check the connection.

In [2]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
Vector dimensions: 256


Write the real articles into a UTF-8 file, with one article per line.
Keep the vector store in a separate folder so its saved row count can guide a restart.

In [3]:
texts = recipe.news(1000)
source = work / "articles.txt"
source.write_text("\n".join(t.replace("\n", " ") for t in texts) + "\n", encoding="utf-8")
store_path = work / "article-vectors"
print("Input articles:", len(texts))

Input articles: 1000


## 2. Save the fingerprint of the file with the vectors

Open the store with `resume=True` and pass the **file itself** to `encode`.
fse hashes the whole file once, which gives its **fingerprint**: its bytes and the order of its lines.
The store keeps the fingerprint, the encoder, and the vector type.
A restart compares them and encodes **only the rows the store does not hold yet**.
Hashing takes CPU time but no model work, and it is the check that stops a restart on a changed file.

In [4]:
def run(sink_class=fse.ShardSink):
    with sink_class(store_path, encoder, shard_rows=200, resume=True) as sink:
        encoder.encode(source, sink=sink, workers=1)
    # first_row: rows saved before this run. published_rows: rows this run added.
    return sink.first_row, sink.published_rows

## 3. Stop, restart, and repeat

A process that dies leaves **whole shards** behind, never half a shard.
To show that, this sink fails when it reaches the third shard, as a killed process would.
The first run keeps what it had saved, the second finishes the file, and the third finds nothing to do.

In [5]:
class Crash(Exception):
    pass


class CrashingSink(fse.ShardSink):
    def commit(self, first_row, rows, **kwargs):
        if first_row >= 400:
            raise Crash("the process died")
        super().commit(first_row, rows, **kwargs)


try:
    run(CrashingSink)
except Crash:
    pass
with fse.VectorStore.open(store_path) as store:
    saved = len(store)
before_restart, added = run()
rerun_before, rerun_added = run()
print({"saved_before_restart": saved, "restart_found": before_restart, "restart_added": added})
print({"rerun_found": rerun_before, "rerun_added": rerun_added})
assert 0 < saved < 1000 and before_restart == saved and added == 1000 - saved
assert rerun_before == 1000 and rerun_added == 0

{'saved_before_restart': 491, 'restart_found': 491, 'restart_added': 509}
{'rerun_found': 1000, 'rerun_added': 0}


The final check compares every stored vector with the vectors an uninterrupted run gives.
They are **identical bit for bit**, not only close.

In [6]:
with fse.VectorStore.open(store_path) as store:
    stored = store.get_vectors()
    store.verify()
fresh = encoder.encode(source, workers=1)
assert stored.view(np.uint32).tobytes() == fresh.view(np.uint32).tobytes()
print("Stored rows:", len(stored), "- every vector equals an uninterrupted run")

Stored rows: 1000 - every vector equals an uninterrupted run


## 4. A changed file is refused

The store saved the fingerprint of the file, so a restart on **another file** stops before it writes a row.
Here one article changes by a single word.

In [7]:
changed = work / "articles-changed.txt"
changed.write_text(source.read_text(encoding="utf-8").replace("the", "THE", 1), encoding="utf-8")
try:
    run_changed = fse.ShardSink(store_path, encoder, shard_rows=200, resume=True)
    with run_changed as sink:
        encoder.encode(changed, sink=sink, workers=1)
except ValueError as error:
    print("Refused:", str(error).split(";")[0].split(str(store_path))[-1].strip())
with fse.VectorStore.open(store_path) as store:
    assert len(store) == 1000

Refused: was written from another source or partition (the source differs)


## 5. Share one big file between processes

`encode(source, sink=sink, shard=(i, n))` takes **part `i` of `n`** of the rows.
The parts move in blocks of 16,384 rows, are disjoint, and together cover the file.
Every process writes a store of its own and can restart on its own.
A file as small as this one has a single block, so one part holds all the rows and the others are empty.

## Result

The store grew to the rows of the file, a restart added only the missing rows, and a rerun added none.
**Resume from the saved prefix**, the uninterrupted sequence of rows starting at zero.
This example stops by an error in the sink, rather than cutting power, so it checks the restart logic and row identity.
A killed process is covered by the tests of fse, which end real processes with `SIGKILL`.

## Before using your own export

Use one writer for each store and keep it on a supported local filesystem.
The source must be something fse can read again: a list, a tuple, an array, or a text file path.
A stream (a generator or a queue) is read once, so `encode(texts, shard=)` cannot resume it. A stream that can be read again from a position resumes with `encode_batches(batches, sink=sink, positions=True)`, as in the [fse behind Kafka recipe](09-fse-behind-kafka.ipynb).
The store keeps the fingerprint of the source, the encoder, and the vector type; a restart with any change is refused.
A crash may leave temporary folders named `.shard-...writing-...` in the store; fse ignores them.
The local store guarantee does not extend to SMB or NFS network storage.

## What next?

- [Watch a folder for new files](05-folder-listener.ipynb).
- [fse API and design](../docs/architecture/store.md)
- [Python file I/O](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)

Back to the [recipe index](README.md) or the [main README](../README.md).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [8]:
recipe.close()